# EXO_04_DARKROOM — Batch Render ATOM-IC

```
╔══════════════════════════════════════════════════════════════════════════════╗
║           PHOTOGRAPHY WING — DARKROOM V1 — BATCH RENDERING                  ║
║                                                                              ║
║   Rendu 1080p @ 128 samples + OIDN (pas 4K direct)                          ║
║   U06 Real-ESRGAN upscale les frames gradées → 4K                           ║
║   Chunks de 300 frames + checkpoint JSON (résiste au timeout 12h)            ║
║   ATOM-IC : Transmutation 1080p → 4K (~2-4h au lieu de 15-45h)              ║
║                                                                              ║
║   Pipeline : scene_ready_*.blend → darkroom_render.py → render_*.png         ║
║   CLI      : EXO_04_DARKROOM.py --drive-root --project-name --resume         ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

**Mode:** Rendu batch headless Blender sur Colab T4. Auto-resume via checkpoint.

## 1. Mount Drive + Configuration

In [ ]:
import os
import sys
import json
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
PROJECT_NAME = "MY_PROJECT"  # <-- Modifier selon le projet
DRIVE_ROOT = Path("/content/drive/MyDrive/EXODUS_V2")
CHUNK_SIZE = 300
PRESET = "darkroom"  # darkroom (1080p/128) | production (4K/256) | preview (1080p/64)

UNIT_ROOT = DRIVE_ROOT / "04_PHOTOGRAPHY_WING"
CODEBASE = UNIT_ROOT / "CODEBASE"
OUT_CAMERA = UNIT_ROOT / "OUT_CAMERA_LOGIC"

sys.path.insert(0, str(CODEBASE))

print(f"Drive Root   : {DRIVE_ROOT}")
print(f"Unit Root    : {UNIT_ROOT}")
print(f"Codebase     : {CODEBASE}")
print(f"OUT_CAMERA   : {OUT_CAMERA}")
print(f"Project      : {PROJECT_NAME}")
print(f"Preset       : {PRESET}")
print(f"Chunk size   : {CHUNK_SIZE}")
print(f"Exists       : {UNIT_ROOT.exists()}")

## 2. Install Blender Headless

In [ ]:
%%bash
# Install Blender 4.0 headless sur Colab
if ! command -v blender &> /dev/null; then
    echo "[DARKROOM] Installation Blender 4.0..."
    apt-get update -qq
    apt-get install -y -qq blender > /dev/null 2>&1
    echo "[DARKROOM] Blender installé"
else
    echo "[DARKROOM] Blender déjà installé"
fi
blender --version
echo "[DARKROOM] GPU disponible :"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "  Pas de GPU NVIDIA"

## 3. Verify Setup

In [ ]:
import shutil

print("=== Pre-flight Check ===")

# Blender
blender_path = shutil.which("blender")
print(f"  Blender        : {blender_path or 'NOT FOUND'}")

# .blend files
blends = sorted(OUT_CAMERA.glob("scene_ready_*.blend")) if OUT_CAMERA.exists() else []
print(f"  .blend trouvés : {len(blends)}")
for b in blends:
    size_mb = b.stat().st_size / (1024 * 1024)
    print(f"    {b.name} ({size_mb:.1f} MB)")

# Preset
from camera_schema import RENDER_PRESETS
print(f"  Preset '{PRESET}' : {'OK' if PRESET in RENDER_PRESETS else 'MISSING'}")
if PRESET in RENDER_PRESETS:
    p = RENDER_PRESETS[PRESET]
    print(f"    Resolution : {p['resolution']}")
    print(f"    Samples    : {p['samples']}")
    print(f"    Denoiser   : {p['denoiser']}")

# Disk space
stat = shutil.disk_usage("/content")
free_gb = stat.free / (1024 ** 3)
print(f"  Espace disque  : {free_gb:.1f} GB libre")

# Checkpoint
ckpt = OUT_CAMERA / "darkroom_checkpoint.json"
if ckpt.exists():
    with open(ckpt) as f:
        ckpt_data = json.load(f)
    print(f"  Checkpoint     : frame {ckpt_data['next_frame']}/{ckpt_data['total_frames']}")
else:
    print(f"  Checkpoint     : Aucun (démarrage frais)")

# Frames already rendered
existing_frames = sorted(OUT_CAMERA.glob("render_*.png")) if OUT_CAMERA.exists() else []
print(f"  Frames rendues : {len(existing_frames)}")

if not blends:
    print("\nERREUR : Aucun scene_ready_*.blend — lancez d'abord U04-A (EXO_04_PRODUCTION)")
elif blender_path is None:
    print("\nERREUR : Blender non installé — relancez la cellule d'installation")
else:
    print(f"\nOK — Prêt pour le rendu ({len(blends)} scènes)")

## 4. Render (avec auto-resume)

In [ ]:
cmd = f'python "{CODEBASE}/EXO_04_DARKROOM.py"'
cmd += f' --drive-root "{DRIVE_ROOT}"'
cmd += f' --project-name "{PROJECT_NAME}"'
cmd += f' --chunk-size {CHUNK_SIZE}'
cmd += f' --preset {PRESET}'
cmd += ' --resume'
cmd += ' -v'

print(f"=== Commande ===")
print(cmd)
print(f"\n=== Exécution ===")
!{cmd}

## 5. Progress Check

In [ ]:
print("=== Progression ===")

# Checkpoint status
ckpt = OUT_CAMERA / "darkroom_checkpoint.json"
if ckpt.exists():
    with open(ckpt) as f:
        data = json.load(f)
    pct = data['frames_rendered'] / data['total_frames'] * 100
    print(f"  Checkpoint : {data['frames_rendered']}/{data['total_frames']} ({pct:.1f}%)")
    print(f"  Next frame : {data['next_frame']}")
    print(f"  Elapsed    : {data['elapsed_seconds'] / 60:.1f} min")
    remaining = data['total_frames'] - data['frames_rendered']
    if data['frames_rendered'] > 0:
        spf = data['elapsed_seconds'] / data['frames_rendered']
        eta = spf * remaining / 60
        print(f"  ETA        : ~{eta:.0f} min ({spf:.1f}s/frame)")
else:
    print("  Pas de checkpoint actif")

# Count frames on disk
frames = sorted(OUT_CAMERA.glob("render_*.png")) if OUT_CAMERA.exists() else []
total_size = sum(f.stat().st_size for f in frames)
print(f"\n  Frames sur disque : {len(frames)}")
print(f"  Taille totale     : {total_size / (1024 * 1024):.1f} MB ({total_size / (1024 ** 3):.2f} GB)")

if frames:
    avg_size = total_size / len(frames)
    print(f"  Taille moyenne    : {avg_size / 1024:.1f} KB/frame")
    print(f"  Première frame    : {frames[0].name}")
    print(f"  Dernière frame    : {frames[-1].name}")

## 6. Verify Output

In [ ]:
print("=== Vérification Output ===")

frames = sorted(OUT_CAMERA.glob("render_*.png")) if OUT_CAMERA.exists() else []
total_size = sum(f.stat().st_size for f in frames)

print(f"  Frames PNG     : {len(frames)}")
print(f"  Taille totale  : {total_size / (1024 ** 3):.2f} GB")

# Check for gaps
if frames:
    numbers = [int(f.stem.split('_')[1]) for f in frames]
    expected = set(range(min(numbers), max(numbers) + 1))
    actual = set(numbers)
    missing = expected - actual
    if missing:
        print(f"  ATTENTION : {len(missing)} frames manquantes")
        print(f"    Exemples : {sorted(missing)[:10]}")
    else:
        print(f"  Séquence   : continue ({min(numbers)}–{max(numbers)})")

# Report
report = OUT_CAMERA / "darkroom_report.json"
if report.exists():
    with open(report) as f:
        data = json.load(f)
    s = data.get('summary', {})
    print(f"\n  Rapport :")
    print(f"    Scènes         : {s.get('total_scenes', 'N/A')}")
    print(f"    Frames rendues : {s.get('total_frames', 'N/A')}")
    print(f"    Temps total    : {s.get('total_elapsed_seconds', 0) / 60:.1f} min")
    print(f"    Moyenne        : {s.get('avg_seconds_per_frame', 'N/A')}s/frame")
else:
    print(f"\n  Rapport : Non trouvé")

# Checkpoint cleanup
ckpt = OUT_CAMERA / "darkroom_checkpoint.json"
if ckpt.exists():
    print(f"\n  ATTENTION : Checkpoint encore présent — rendu peut-être incomplet")
    print(f"  Relancez la cellule Render pour reprendre")
else:
    print(f"\n  Checkpoint : supprimé (rendu complet)")

print("\n" + "=" * 60)
print("   DARKROOM — VÉRIFICATION TERMINÉE")
print("=" * 60)